# Lumen Clip Kaggle GPU backend
Run cells in order. Keep this notebook running while the website generates videos.
Enable Settings -> Accelerator -> GPU P100 or T4 first.

## 1. Install dependencies

In [ ]:
import subprocess, sys
pkgs = ['diffusers>=0.29.0','transformers>=4.41.0','accelerate>=0.31.0','safetensors>=0.4.3','fastapi>=0.111.0','uvicorn>=0.30.0','imageio>=2.34.0','imageio-ffmpeg>=0.5.1','opencv-python-headless>=4.10.0','pydantic>=2.7.0']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
print('deps ready')

## 2. Check CUDA/GPU

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('Enable Settings -> Accelerator -> GPU, then Restart session.')
print('gpu name', torch.cuda.get_device_name(0))
print('vram_gb', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 3. Download latest GitHub files

In [ ]:
from pathlib import Path
import urllib.request, sys
REPO = 'https://raw.githubusercontent.com/sgue19000/t2v-kaggle-webapp/main/kaggle'
workdir = Path('/kaggle/working')
for name in ('generator.py', 'server.py'):
    urllib.request.urlretrieve(f'{REPO}/{name}', workdir / name)
    print('updated', name)
if str(workdir) not in sys.path:
    sys.path.insert(0, str(workdir))

## 4. Load the model

In [ ]:
from generator import load_pipeline, model_info, gpu_report
print(gpu_report())
load_pipeline()
print(model_info())

## 5. Test a tiny local generation

In [ ]:
from pathlib import Path
from generator import generate_video
demo = Path('/kaggle/working/outputs/demo.mp4')
path = generate_video({'prompt': 'A golden retriever running through tall grass at sunrise', 'num_frames': 8, 'height': 256, 'width': 256, 'fps': 8, 'steps': 15, 'guidance_scale': 9, 'seed': 42}, out_path=demo)
print('wrote', path, 'bytes', path.stat().st_size)

## 6. Start FastAPI

In [ ]:
import os, time, threading
from pathlib import Path
os.environ['T2V_OUTPUT_DIR'] = '/kaggle/working/outputs'
os.environ['T2V_SKIP_PRELOAD'] = '1'
Path(os.environ['T2V_OUTPUT_DIR']).mkdir(parents=True, exist_ok=True)
def run_api():
    import uvicorn
    from server import app
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')
threading.Thread(target=run_api, daemon=True).start()
time.sleep(3)
print('FastAPI listening on 0.0.0.0:8000')

## 7. Start Cloudflare Quick Tunnel

In [ ]:
import time, subprocess, re
from pathlib import Path
cf = Path('/kaggle/working/cloudflared')
if not cf.exists():
    subprocess.check_call(['wget', '-q', '-O', str(cf), 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'])
    cf.chmod(0o755)
log_path = Path('/kaggle/working/tunnel.log')
log = open(log_path, 'w')
subprocess.Popen([str(cf), 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'], stdout=log, stderr=subprocess.STDOUT)
url = None
for _ in range(45):
    time.sleep(1)
    text = log_path.read_text(errors='ignore')
    found = re.findall(r'https://[-a-z0-9.]+trycloudflare.com', text)
    if found:
        url = found[-1]
        break
if not url:
    raise SystemExit('Tunnel URL missing. Re-run this cell.')
print('='*40)
print('PUBLIC API URL:')
print(url)
print('='*40)

## 8. Public HTTPS URL is printed above

## 9. Test /health

In [ ]:
import json, urllib.request
print(json.dumps(json.load(urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=30)), indent=2))

## 10. Test /generate

In [ ]:
import json, time, urllib.request
req = urllib.request.Request('http://127.0.0.1:8000/generate', data=json.dumps({'prompt': 'A cinematic futuristic city at night, flying cars, rain', 'num_frames': 8, 'width': 256, 'height': 256, 'fps': 8, 'steps': 15, 'guidance_scale': 9, 'seed': 12345}).encode(), headers={'Content-Type': 'application/json'}, method='POST')
started = json.load(urllib.request.urlopen(req, timeout=30))
print(started)
job_id = started['job_id']
for _ in range(120):
    snap = json.load(urllib.request.urlopen(f'http://127.0.0.1:8000/status/{job_id}', timeout=30))
    print(snap.get('status'), snap.get('progress'), snap.get('message') or snap.get('error'))
    if snap.get('status') in ('completed', 'failed'):
        break
    time.sleep(5)
print(snap)
globals()['LAST_JOB_ID'] = job_id

## 11. Display the generated MP4

In [ ]:
from IPython.display import Video, display
from pathlib import Path
job_id = globals().get('LAST_JOB_ID')
paths = [Path(f'/kaggle/working/outputs/{job_id}.mp4')] if job_id else []
paths.append(Path('/kaggle/working/outputs/demo.mp4'))
shown = False
for p in paths:
    if p.exists() and p.stat().st_size > 1024:
        print(p, p.stat().st_size)
        display(Video(str(p), embed=True))
        shown = True
        break
if not shown:
    print('No MP4 found')